# Imports & Setup

In [1]:
import torch
import wandb
import pandas as pd
from pathlib import Path


## Helper to inspect checkpoint

In [2]:
def load_checkpoint_info(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = ckpt["config"]

    return {
        "path": ckpt_path,
        "file": Path(ckpt_path).name,
        "run_id": cfg["logging.run_id"],
        "project": cfg["logging.project"],
        "best_epoch": ckpt["best_epoch"],
        "best_metric": ckpt["best_metric"],
        "config": cfg
    }


## Get W&B metrics for a given run and epoch

In [3]:
def extract_metrics_from_wandb(run_id, project, best_epoch):
    api = wandb.Api()
    run = api.run(f"{project}/{run_id}")

    hist = run.history()  # full history
    row = hist[hist["_step"] == best_epoch]

    if len(row) == 0:
        print(f"No row found for epoch {best_epoch}")
        return None

    # Take the first (and only) matching row
    row = row.iloc[0].to_dict()
    return row


# Metrics we want in the comparison table

In [4]:
METRICS_OF_INTEREST = [
    "Binary-Cell-Dice-Mean/Validation",
    "Binary-Cell-Jacard-Mean/Validation",
    "mPQ/Validation",
    "epithelial-PQ/Validation",
    "lymphocyte-PQ/Validation",
    "macrophage-PQ/Validation",
    "neutrophil-PQ/Validation",
    "bPQ/Validation",
    "hv_map_mse/Validation",
    "hv_map_msge/Validation",
]


## Compare Checkpoints and Create a Df

In [5]:
def compare_checkpoints(
    configs,
    metrics=METRICS_OF_INTEREST,
    show_best_epoch=False,
    float_precision=4
):
    rows = []

    for cfg_entry in configs:
        ckpt_path = cfg_entry["path"]
        name = cfg_entry.get("name", Path(ckpt_path).stem)
        patch_size = cfg_entry.get("patch_size", None)

        info = load_checkpoint_info(ckpt_path)

        metrics_row = extract_metrics_from_wandb(
            run_id=info["run_id"],
            project=info["project"],
            best_epoch=info["best_epoch"],
        )

        row_data = {
            "name": name,
            "checkpoint": info["file"],
            "patch_size": patch_size,
            "best_metric(IoU)": info["best_metric"],
        }

        if show_best_epoch:
            row_data["best_epoch"] = info["best_epoch"]

        for m in metrics:
            row_data[m] = metrics_row.get(m, None)

        rows.append(row_data)

    df = pd.DataFrame(rows)

    # Round floats
    df = df.round(float_precision)

    return df


In [6]:
checkpoint_paths = [
    #"/path/to/cellvit128.ckpt",
]

checkpoint_configs = [
    {
        "path":"/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local_berk/2025-12-05T191217_First film try same with patch size 256/checkpoints/model_best.pth",
        "name": "FiLM-256",
        "patch_size": 256,
    },
    {
        "path": "/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p128_pannuke/logs_local/2025-12-04T002122_First film try/checkpoints/model_best.pth",
        "name": "FiLM-128",
        "patch_size": 128,
    },
    {
        "path":  "/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p128_pannuke/logs_local/2025-12-03T180258_tcga_finetune_128/checkpoints/model_best.pth",
        "name": "CellViT-128",
        "patch_size": 128,
    },
]

"""{
        "path": ,
        "name": "CellViT-256",
        "patch_size": 256,
    },""" #TODO Add this part


df = compare_checkpoints(
    configs=checkpoint_configs,
    metrics=METRICS_OF_INTEREST,
    show_best_epoch=False,
    float_precision=3,
)

df


,name,checkpoint,patch_size,best_metric(IoU),Binary-Cell-Dice-Mean/Validation,Binary-Cell-Jacard-Mean/Validation,mPQ/Validation,epithelial-PQ/Validation,lymphocyte-PQ/Validation,macrophage-PQ/Validation,neutrophil-PQ/Validation,bPQ/Validation,hv_map_mse/Validation,hv_map_msge/Validation
0,FiLM-256,model_best.pth,256,0.688,0.819,0.709,0.602,0.655,0.535,0.424,0.421,0.683,0.024,0.207
1,FiLM-128,model_best.pth,128,0.618,0.787,0.678,0.555,0.580,0.536,0.377,0.415,0.613,0.027,0.223
2,CellViT-128,model_best.pth,128,0.621,0.786,0.675,0.562,0.588,0.521,0.381,0.475,0.616,0.029,0.220


## Save df as pickle

In [21]:
from pathlib import Path

comparison_dir = Path("../comparison_dfs")
comparison_dir.mkdir(exist_ok=True)


In [24]:
def save_comparison_df(df, name, folder=comparison_dir):
    """
    Saves comparison dataframe as a pickle file.
    Args:
        df (pd.DataFrame): The comparison table.
        name (str): File name without extension.
    """
    path = Path(folder) / f"{name}.pkl"
    df.to_pickle(path)
    print(f"Saved comparison df to {path}")


In [29]:
df_name = "film_vs_cellvit"
save_comparison_df(df, df_name)

Saved comparison df to ../comparison_dfs/film_vs_cellvit.pkl


## Read saved df

In [30]:
def load_comparison_df(name, folder=comparison_dir):
    """
    Loads a saved comparison dataframe from pickle.
    """
    path = Path(folder) / f"{name}.pkl"
    df = pd.read_pickle(path)
    print(f"📄 Loaded df from {path}")
    return df


In [31]:
df_loaded = load_comparison_df("film_vs_cellvit")
df_loaded


📄 Loaded df from ../comparison_dfs/film_vs_cellvit.pkl


,name,checkpoint,patch_size,best_metric(IoU),Binary-Cell-Dice-Mean/Validation,Binary-Cell-Jacard-Mean/Validation,mPQ/Validation,epithelial-PQ/Validation,lymphocyte-PQ/Validation,macrophage-PQ/Validation,neutrophil-PQ/Validation,bPQ/Validation,hv_map_mse/Validation,hv_map_msge/Validation
0,FiLM-256,model_best.pth,256,0.688,0.819,0.709,0.602,0.655,0.535,0.424,0.421,0.683,0.024,0.207
1,FiLM-128,model_best.pth,128,0.618,0.787,0.678,0.555,0.580,0.536,0.377,0.415,0.613,0.027,0.223
2,CellViT-128,model_best.pth,128,0.621,0.786,0.675,0.562,0.588,0.521,0.381,0.475,0.616,0.029,0.220


## Merge dfs

In [28]:
def merge_comparison_dfs(df_list, how="outer"):
    """
    Merge a list of comparison dataframes row-wise.
    Args:
        df_list (list[pd.DataFrame])
        how (str): 'outer' or 'inner'
    """
    merged = pd.concat(df_list, axis=0, join=how, ignore_index=True)
    print(f"Merged {len(df_list)} dataframes")
    return merged


In [32]:
#Example
df1 = load_comparison_df("film_vs_cellvit")
#df2 = load_comparison_df("cellvit_p256_baseline")

df_list = [df1]

merged_df = merge_comparison_dfs(df_list)
merged_df

#To save, give a name
df_name = "film_vs_cellvit_2"
save_comparison_df(merged_df, df_name)


📄 Loaded df from ../comparison_dfs/film_vs_cellvit.pkl
Merged 1 dataframes
Saved comparison df to ../comparison_dfs/film_vs_cellvit_2.pkl
